# Guidelines for use of MAST_tools

Notebook demostrating the utilities in MAST_tools modules.

In [ ]:
import sys
sys.path.insert(1, '../src')

import numpy as np
from pprint import pprint
from MAST_tools.utils.store_utils import MASTStorageManager
from MAST_tools.utils.signal_utils import MASTSignalManager
from MAST_tools.utils.plotting_utils import MASTPlottingManager


# Store utilities

## Creation of store manager

In [ ]:
store_manager = MASTStorageManager()

### List all available shot IDs

In [ ]:
all_shots_ids = store_manager.list_all_shots(
    local=True
)

pprint(all_shots_ids)


### List all sources

In [ ]:
all_sources = store_manager.get_all_sources(
    shot_ids=[30421],  # all_shots_ids[0]
    local=False
)

try:
    pprint(all_sources)
    print("\n\nall_sources.keys:\n")
    pprint(all_sources.keys())
except Exception as e:
    print(e)
        

### List all signals

In [ ]:
all_signals = store_manager.get_all_signals(
    shot_ids=[30421],  # Use None for the entire dataset.
    local=False
)

pprint(all_signals)


### Make group directly from shot_id

In [ ]:
store_ = store_manager.make_shot_store(
    shot_info={
        "shot_id": 30421,
        "local": False
    }
)
group_from_store = store_manager.make_shot_group(data_origin=store_)
print(f"group_from_store.groups (group from store): {group_from_store.groups}\n")


### Make group from existing store object

In [ ]:
group_from_shot_info = store_manager.make_shot_group(
    data_origin={
        "shot_id": 30421,
        "local": False
    }
)
print(f"group_from_shot_info.groups (group from shot info): {group_from_shot_info.groups}\n")

if False:
    pprint(store_manager.get_all_signals_in_group(group=group_from_shot_info))

### Check signal availability

In [ ]:
dict_target_signals = {
    "thomson_scattering": ["n_e"],
    "spectrometer_visible": ["filter_spectrometer_bes_voltage"],
    "summary": ["power_nbi", "ip"],
}

print("\nSignals to be checked for simultaneous availability across all shots:")
pprint(dict_target_signals)
    
filtered_ids = store_manager.list_shots_by_signal_availability(
    required_signals=dict_target_signals
)

print(f"\nfiltered_ids ({len(filtered_ids)} shots):")
pprint(filtered_ids)

# Signal utilities

### Creation of signal manager

In [ ]:
signal_manager = MASTSignalManager()

### Get signal values from existing store 

In [ ]:
# First create a store from shot info
store_from_shot_info = store_manager.make_shot_store(
    shot_info={
        "shot_id": 30421,
        "local": False
    }
)

# Then get signal values from that store
signal_values = signal_manager.get_signal_values(
    data_origin=store_from_shot_info,
    source_name="magnetics",
    signal_name="flux_loop_flux"
)

print("Signal values:\n")
pprint(signal_values)
    

### Get signal values from shot info

In [ ]:
# Get signal values directly from shot info

shot_info = {
    "shot_id": 30421,
    "local": False
}
# source_name, signal_name = ("thomson_scattering", "n_e")
source_name, signal_name = ("magnetics", "flux_loop_flux")

signal_values = signal_manager.get_signal_values(
    data_origin=shot_info,
    source_name=source_name,
    signal_name=signal_name
)

print("Signal values:\n")
pprint(signal_values)

if False:
    np.save(f"{source_name}__{signal_name}__{shot_info['shot_id']}", signal_values)


# Plotting utilities

## Creation of managers

In [ ]:
plotting_manager = MASTPlottingManager()
signal_manager = MASTSignalManager()

## Settings for tests

In [ ]:
%matplotlib notebook
# %matplotlib inline

# Create a group to plot from
group_from_shot_info = store_manager.make_shot_group(
    data_origin={
        "shot_id": 30421,
        "local": False
    }
)

available_sources = list(group_from_shot_info.keys())
target_source = 'summary'  # available_sources[0]

available_signals = list(group_from_shot_info[target_source].keys())

# Create source profiles from target store
source_profiles = signal_manager.get_source_profiles(
    data_origin=group_from_shot_info.store,
    source_name=target_source
)

source_profiles


## Plotting examples

### Plot single profile (from DataArray)

In [ ]:
# Select target signal from available signals
target_signal = available_signals[0]

# Plot signal
plotting_manager.plot_1d_profiles(
    profiles=source_profiles[target_signal],
    fig_size=[8, 4]
)


### Plot group of profiles (from Dataset)

In [ ]:
%matplotlib inline

plotting_manager.plot_1d_profiles(
    profiles=source_profiles,
    fig_size=[8, 8]
)

### Plot target signal from store

In [ ]:
%matplotlib inline

plotting_manager.plot_signal(
    data_origin=group_from_shot_info.store,
    source_name="magnetics",
    signal_name="ip",
    fig_size=[8, 4]
)

### Plot target signal from shot info

In [ ]:
%matplotlib inline

plotting_manager.plot_signal(
    data_origin={"shot_id": 30421, "level": 2, "test_data": False, "local": False, "via_parquet": False},
    source_name="thomson_scattering",  # "magnetics",
    signal_name="n_e",  # "ip",
    fig_size=[8, 4]
)

### Plot target group from store

In [ ]:
%matplotlib inline

plotting_manager.plot_group(
    data_origin=group_from_shot_info.store,
    source_name="magnetics",
    # fig_size=[8, 12]
)

### Plot target group from shot info

In [ ]:
%matplotlib inline

plotting_manager.plot_group(
    data_origin={
        "shot_id": 30421,
        "local": False
    },
    source_name="magnetics",
    # fig_size=[8, 4]
)

### Plot specific groups

In [ ]:
%matplotlib inline

plotting_manager.plot_plasma_current(data_origin=group_from_shot_info.store)
    

In [ ]:
%matplotlib inline

plotting_manager.plot_power_nbi(data_origin=group_from_shot_info.store)
    

In [ ]:
%matplotlib inline

plotting_manager.plot_magnetics(data_origin=group_from_shot_info.store)
    

In [ ]:
%matplotlib inline

plotting_manager.plot_spectrometer(data_origin=group_from_shot_info.store)
    

In [ ]:
%matplotlib inline

plotting_manager.plot_charge_exchange(data_origin=group_from_shot_info.store)
    

In [ ]:
%matplotlib inline

plotting_manager.plot_thomson_scattering(data_origin=group_from_shot_info.store)
    